In [0]:
import uuid
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField, StringType, TimestampType, LongType
)

CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
SCHEMA  = "cyber_threat"

BRONZE_NVD  = f"{CATALOG}.{SCHEMA}.bronze_nvd"
BRONZE_EPSS = f"{CATALOG}.{SCHEMA}.bronze_epss"
BRONZE_KEV  = f"{CATALOG}.{SCHEMA}.bronze_kev"
SILVER_NVD  = f"{CATALOG}.{SCHEMA}.silver_nvd"
SILVER_EPSS = f"{CATALOG}.{SCHEMA}.silver_epss"
SILVER_KEV  = f"{CATALOG}.{SCHEMA}.silver_kev"
LOG_TABLE   = f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.quarantine_records"

print("Catalog:", CATALOG, "| Schema:", SCHEMA)

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    run_id          STRING,
    layer           STRING,
    load_type       STRING,
    source_param    STRING,
    target_table    STRING,
    start_time      TIMESTAMP,
    end_time        TIMESTAMP,
    status          STRING,
    rows_inserted   BIGINT,
    rows_updated    BIGINT,
    error_message   STRING,
    load_timestamp  TIMESTAMP
) USING DELTA
""")

In [0]:
log_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("load_type", StringType(), True),
    StructField("source_param", StringType(), True),
    StructField("target_table", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("error_message", StringType(), True),
    StructField("load_timestamp", TimestampType(), True),
])

def new_run_id():
    return str(uuid.uuid4())

def write_log(run_id, layer, load_type, source_param, target_table,
              start_time, status, rows_inserted=0, rows_updated=0, error_message=None):
    end_time = datetime.now()
    row = [(run_id, layer, load_type, str(source_param), target_table,
            start_time, end_time, status,
            int(rows_inserted), int(rows_updated),
            error_message, end_time)]
    spark.createDataFrame(row, schema=log_schema) \
         .write.format("delta").mode("append").saveAsTable(LOG_TABLE)